# Test-Time Augmentation Views: What They Cost and What They Buy

`RFDETRInference.predict(images, tta=TTAConfig(...))` runs the same network on several *views* of each image, maps every view's detections back to the image, and averages the matching ones. A view is a copy of the image that is moved by a fraction of a backbone patch (`shift`), mirrored (`flip`) or resized (`zoom`). On a CUDA device the views of one canvas size run as one forward pass, so `V` views of one image cost about what a batch of `V` images costs, plus the work of building the views and fusing the results.

This notebook answers four questions on your own hardware, for every model size:

1. How much box AP do `V` views add on full COCO val2017, and how far does that gain move between disjoint folds of the data?
2. What does a call cost in latency and peak memory for one image with `V` views (`V` = 1, 2, 4, 6, 8)? Latency is also measured for a batch of `B` images, so views and batch images compare on the same rows axis.
3. How much of the cost is the work around the network (placing the views, mapping back, decoding, fusing) and how much is the forward pass?
4. Does a view recipe other than shifts (mirror, zoom) buy more AP for the same `V`, and do cheap runtime switches speed the calls up without moving AP?

> **Note:** Nothing here is a published result. Every number comes from the run you start, on the device and package versions printed in the footer at the end. Copy numbers into other docs only together with that footer.

Install the package with the COCO scoring backends (`train`), the plotting dependencies (`visual`) and the Plus models (`plus`). Until the views land in a release, the first line installs the branch that has them. Restart the kernel afterwards so the recorded versions are the ones in use.

In [ ]:
# Remove the git URL and keep the extras once a release ships `TTAConfig`.
!pip install -U "rfdetr[train,visual,plus] @ git+https://github.com/roboflow/rf-detr.git@feat/tta-views" "rfdetr_plus>=1.2.0"

Keep imports together so every dependency of the measurement path is visible in one place.

In [ ]:
import dataclasses
import importlib.metadata
import itertools
import json
import platform
import time
from collections import defaultdict
from collections.abc import Iterator
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from typing import NamedTuple
from unittest import mock

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import supervision as sv
import torch
from IPython.display import display
from PIL import Image
from torchvision.transforms.functional import resize, to_tensor
from tqdm.auto import tqdm

import rfdetr
from rfdetr import RFDETRInference
from rfdetr._prediction import core as prediction_core
from rfdetr._prediction.tta import TTAConfig, view_count
from rfdetr.datasets._develop import _download_and_extract
from rfdetr.export._benchmark import (
    CocoValSubset,
    cpu_brand,
    evaluate_coco_predictions,
    fetch_coco_val2017,
    measure_latency,
)

## Choose what to run

The defaults cover every released detection size, the Plus models included while `ACCEPT_PLATFORM_MODEL_LICENSE = True`, and score all 5000 COCO val2017 images. `VIEWS` counts views per image: 1 is the plain prediction and the baseline every other row is compared with. A full run takes hours on one GPU; every row is written to disk as soon as it exists, so an interrupted run resumes. For a quick check lower `N_AP_IMAGES`, shorten `MODELS` and `VIEWS`.

> **Note:** PML 1.0 covers the Plus detection models (Atto, Femto, Pico, XLarge and 2XLarge): commercial use is included when you deploy on Roboflow, and commercial self-hosting requires the Enterprise add-on. Apache 2.0 covers the package, Nano–Large detection, every Seg size and Keypoint: commercial, closed-source and self-hosted use is included without a license fee or an obligation to open-source your code. Keep `ACCEPT_PLATFORM_MODEL_LICENSE = True` only if you accept the Plus model terms; set it to `False` to measure the Apache 2.0 models alone.

In [ ]:
MODELS = ("RFDETRNano", "RFDETRSmall", "RFDETRMedium", "RFDETRLarge")
PLUS_MODELS = ("RFDETRAtto", "RFDETRFemto", "RFDETRPico", "RFDETRXLarge", "RFDETR2XLarge")
ACCEPT_PLATFORM_MODEL_LICENSE = True
# Smallest to largest, so plots and tables list the sizes in the order a reader expects.
SIZE_ORDER = ("Atto", "Femto", "Pico", "Nano", "Small", "Medium", "Large", "XLarge", "2XLarge")
if ACCEPT_PLATFORM_MODEL_LICENSE:
    MODELS = tuple(sorted(MODELS + PLUS_MODELS, key=lambda name: SIZE_ORDER.index(name.removeprefix("RFDETR"))))

# Views per image. 1 is the plain prediction.
VIEWS = (1, 2, 4, 6, 8)
# Images scored for AP: None scores all 5000 val2017 images.
N_AP_IMAGES = None
# AP is also scored on this many disjoint folds, to show how much a gain moves between parts of the data.
N_FOLDS = 5
# Images per call while scoring AP. AP does not depend on it; it only keeps the scoring pass short.
AP_BATCH = 8
# A confidence threshold near zero traces the whole precision-recall curve; 0.5 is a typical serving value.
AP_THRESHOLD = 0.001
LATENCY_THRESHOLD = 0.5
# Distinct images the latency calls cycle through, so no cache flatters the timing.
N_LATENCY_IMAGES = 64
WARMUP = 10
RUNS = 50
# Calls averaged for the per-stage timing, which synchronizes the device around every stage.
STAGE_RUNS = 20
N_STABILITY_IMAGES = 100
# Models that also run the other view recipes, and the images and models of the runtime-switch check.
RECIPE_MODELS = ("RFDETRNano", "RFDETRSmall", "RFDETRLarge")
SWITCH_MODELS = ("RFDETRNano", "RFDETRSmall", "RFDETRLarge")
N_SWITCH_AP_IMAGES = 1000

COCO_ROOT = Path("coco")
WORK_DIR = Path("tta-views-run")
RESULTS_PATH = WORK_DIR / "results.jsonl"

Describe one arm of the sweep: a recipe name, how many views it runs and the `TTAConfig` that produces them. `shift` is the main recipe; the others compare what the same number of views buys when they are mirrored or zoomed instead. Views are additive, so `shift=2` with two zooms is four views, and `view_count` confirms each arm runs the number of views it claims.

In [ ]:
class Arm(NamedTuple):
    """One view recipe at one view count."""

    recipe: str
    views: int
    config: TTAConfig | None


# `None` keeps V=1 on the plain path, the baseline every other row is compared with.
PLAIN = Arm("shift", 1, None)
SHIFT_ARMS = [Arm("shift", v, TTAConfig(shift=v) if v > 1 else None) for v in VIEWS]
OTHER_ARMS = [
    Arm("flip", 2, TTAConfig(flip=True)),
    Arm("shift+flip", 4, TTAConfig(shift=2, flip=True)),
    Arm("shift+flip", 8, TTAConfig(shift=4, flip=True)),
    Arm("zoom", 2, TTAConfig(zoom=(1.25,))),
    Arm("zoom", 4, TTAConfig(zoom=(0.85, 1.25, 1.5))),
    Arm("shift+zoom", 4, TTAConfig(shift=2, zoom=(1.25, 1.5))),
    Arm("shift+zoom", 8, TTAConfig(shift=4, zoom=(0.85, 1.15, 1.3, 1.5))),
]
for arm in SHIFT_ARMS + OTHER_ARMS:
    assert arm.config is None or view_count(arm.config) == arm.views, arm

Reject unusable settings before any download. The shift count of a model is bounded by its patch size, so that part is checked after each model loads.

In [ ]:
assert MODELS and VIEWS, "Select at least one model and view count."
assert VIEWS[0] == 1, "List 1 first: V=1 is the baseline the other rows are compared with."
assert all(isinstance(v, int) and v >= 1 for v in VIEWS), "View counts are integers >= 1."
assert ACCEPT_PLATFORM_MODEL_LICENSE or not set(MODELS) & set(PLUS_MODELS), (
    "Plus models require ACCEPT_PLATFORM_MODEL_LICENSE = True after accepting PML 1.0."
)
assert N_AP_IMAGES is None or N_AP_IMAGES >= N_FOLDS, "Each fold needs at least one image."
WORK_DIR.mkdir(parents=True, exist_ok=True)

## Prepare COCO val2017

Download the image archive in one transfer (5000 single downloads can hit socket timeouts); the completion check requires 5000 nonempty JPEG files. Skipped when the folder is already complete.

In [ ]:
_download_and_extract(
    "http://images.cocodataset.org/zips/val2017.zip",
    COCO_ROOT,
    is_complete=lambda: sum(path.stat().st_size > 0 for path in (COCO_ROOT / "val2017").glob("*.jpg")) == 5000,
)

Fetch the annotations and pick the images to score. A subset is seeded, so every row scores the same images and a difference between rows is not a difference in data.

In [ ]:
subset = fetch_coco_val2017(COCO_ROOT, n_images=N_AP_IMAGES)
print(f"Scoring {len(subset.image_ids)} COCO val2017 images per row.")

## Record provenance

Record the device, package versions and UTC start date once, so every table and plot can be traced to the run that made it.

In [ ]:
device_kind = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device_name = torch.cuda.get_device_name() if device_kind == "cuda" else cpu_brand()
direct_url = json.loads(importlib.metadata.distribution("rfdetr").read_text("direct_url.json") or "{}")


def package_version(name: str) -> str | None:
    """Return the installed version of ``name``, or ``None`` when it is not installed."""
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


provenance = {
    "device": f"{device_kind}: {device_name}",
    "rfdetr": package_version("rfdetr"),
    "commit": direct_url.get("vcs_info", {}).get("commit_id"),
    "rfdetr_plus": package_version("rfdetr_plus"),
    "torch": torch.__version__,
    "supervision": package_version("supervision"),
    "faster_coco_eval": package_version("faster-coco-eval"),
    "python": platform.python_version(),
    "started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
print(json.dumps(provenance, indent=2))

Views batch on CUDA only; on MPS and CPU each view is its own forward pass. Say so up front, since it changes what the latency plots mean.

In [ ]:
if device_kind != "cuda":
    print(
        f"Device is {device_kind}: views run one forward pass each here, so latency grows about linearly with the "
        "view count. Read the latency plots as the worst case, not as the batched cost."
    )

## Load the images the timing and stability probes reuse

Latency calls cycle through distinct images. Read them once, up front, so disk reads stay out of the timed region.

In [ ]:
# COCO names each file after its zero-padded image id.
image_paths = {image_id: subset.images_dir / f"{image_id:012d}.jpg" for image_id in subset.image_ids}
pool_ids = subset.image_ids[: max(N_LATENCY_IMAGES, N_STABILITY_IMAGES)]
pool = [Image.open(image_paths[image_id]).convert("RGB") for image_id in pool_ids]
print(f"Loaded {len(pool)} images.")

## Define the measurements

Each measurement is one small function, so the loop that runs them stays readable. All of them go through the public `RFDETRInference.predict(..., tta=...)`, the call users make.

Box AP first. The predictor scores a whole list of image paths in batches and the predictions are kept, so the same run is scored on all images and on each disjoint fold without predicting twice. `fold_box_ap` is what the paired gain and its spread are built from.

In [ ]:
def score_ap(predictor: RFDETRInference, arm: Arm, images: CocoValSubset) -> dict[str, object]:
    """Return box AP, AP50 and the per-fold box AP of ``arm`` on the COCO subset ``images``."""
    ids = list(images.image_ids)
    detections = predictor.predict(
        [str(image_paths[i]) for i in ids],
        threshold=AP_THRESHOLD,
        include_source_image=False,
        batch=AP_BATCH,
        tta=arm.config,
    )
    stored = dict(zip(ids, detections, strict=True))
    whole = evaluate_coco_predictions(stored, images, "detect")
    # Interleaved folds, so every fold sees the whole range of the data.
    folds = [ids[k::N_FOLDS] for k in range(N_FOLDS)]
    fold_ap = [evaluate_coco_predictions(stored, images, "detect", image_ids=fold).box_ap for fold in folds]
    return {"box_ap": whole.box_ap, "box_ap50": whole.box_ap50, "fold_box_ap": fold_ap}

Latency next. A call takes `B` distinct images and returns after fusion, so the timing covers preprocessing, the forward pass, mapping back and fusing: the cost a user pays per call. Both the call time and the per-image time are kept, so a row with `B` images and a row with `V` views compare on one rows axis.

In [ ]:
def time_call(predictor: RFDETRInference, arm: Arm, batch_size: int) -> dict[str, float]:
    """Return the median and 95th-percentile time of one call of ``batch_size`` images, in milliseconds."""
    batches = itertools.cycle(
        [pool[i : i + batch_size] for i in range(0, N_LATENCY_IMAGES - batch_size + 1, batch_size)]
    )

    def call() -> object:
        return predictor.predict(next(batches), threshold=LATENCY_THRESHOLD, include_source_image=False, tta=arm.config)

    timing = measure_latency(
        call, label=f"{arm.recipe} V={arm.views} B={batch_size}", device=device_kind, warmup=WARMUP, runs=RUNS
    )
    return {"call_ms": timing.median_ms, "call_p95_ms": timing.p95_ms}

Peak memory is read from the CUDA allocator's own high-water mark around one call, in MiB above what was allocated before it. The sampler in `measure_memory` reads free device memory, which the caching allocator hides, so it reads 0 for calls that reuse cached blocks.

In [ ]:
def peak_memory_mb(predictor: RFDETRInference, arm: Arm, batch_size: int) -> float:
    """Return the peak extra CUDA memory of one call of ``batch_size`` images in MiB, or NaN off CUDA."""
    if device_kind != "cuda":
        return float("nan")
    images = pool[:batch_size]
    # One untimed call first, so one-off allocations (kernels, cuDNN workspaces) are not charged to the measured call.
    predictor.predict(images, threshold=LATENCY_THRESHOLD, include_source_image=False, tta=arm.config)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    before = torch.cuda.memory_allocated()
    predictor.predict(images, threshold=LATENCY_THRESHOLD, include_source_image=False, tta=arm.config)
    torch.cuda.synchronize()
    return (torch.cuda.max_memory_allocated() - before) / 2**20

Stage timing answers the overhead question directly. It times the forward pass with hooks on the network and each step around it (placing the views, mapping boxes back, decoding detections, fusing) by wrapping the module-level functions of the prediction pipeline, synchronizing the device at both ends of every stage. This is a diagnostic: it reaches into private functions and its synchronization makes the total a little longer than an unsynchronized call, so quote stage *shares* from here and totals from the latency rows.

In [ ]:
STAGES = {
    "place_view": "place views",
    "map_to_image": "map back",
    "_shift_candidates": "decode",
    "_fuse_shifted_image": "fuse",
}


def synchronize() -> None:
    """Wait for the device, so a timer reads the work it started and not what is still queued."""
    if device_kind == "cuda":
        torch.cuda.synchronize()
    elif device_kind == "mps":
        torch.mps.synchronize()


@contextmanager
def stage_timers(model: rfdetr.RFDETR) -> Iterator[dict[str, float]]:
    """Accumulate synchronized seconds per stage of every prediction call inside the block."""
    totals: dict[str, float] = defaultdict(float)

    def timed(name: str, function):
        def wrapper(*args, **kwargs):
            synchronize()
            start = time.perf_counter()
            result = function(*args, **kwargs)
            synchronize()
            totals[name] += time.perf_counter() - start
            return result

        return wrapper

    started: list[float] = []

    def forward_pre(*_: object) -> None:
        synchronize()
        started.append(time.perf_counter())

    def forward_post(*_: object) -> None:
        synchronize()
        totals["forward"] += time.perf_counter() - started.pop()

    module = model.model.model
    handles = [module.register_forward_pre_hook(forward_pre), module.register_forward_hook(forward_post)]
    patches = {name: timed(name, getattr(prediction_core, name)) for name in STAGES}
    try:
        with mock.patch.multiple(prediction_core, **patches):
            yield totals
    finally:
        for handle in handles:
            handle.remove()


def time_stages(model: rfdetr.RFDETR, predictor: RFDETRInference, arm: Arm) -> dict[str, float]:
    """Return the mean milliseconds per call of each stage at one image per call, plus the unattributed rest."""
    image = pool[:1]
    for _ in range(3):
        predictor.predict(image, threshold=LATENCY_THRESHOLD, include_source_image=False, tta=arm.config)
    with stage_timers(model) as totals:
        synchronize()
        start = time.perf_counter()
        for i in range(STAGE_RUNS):
            predictor.predict(
                pool[i % len(pool) : i % len(pool) + 1],
                threshold=LATENCY_THRESHOLD,
                include_source_image=False,
                tta=arm.config,
            )
        synchronize()
        total = (time.perf_counter() - start) / STAGE_RUNS * 1000
    stages = {STAGES.get(name, name): seconds / STAGE_RUNS * 1000 for name, seconds in totals.items()}
    stages["other"] = total - sum(stages.values())
    stages["total"] = total
    return stages

Steadiness last. Shift views exist because a score depends a little on where an object sits inside the backbone's patch grid. The probe moves each image down and right by a few pixels inside one patch, predicts every moved copy, follows each reference detection through the copies, and reports how much its score varies. Lower is steadier. The image is first resized to the model's square input, so a shift in pixels is a shift in model-input pixels; each copy loses a few border pixels, which is acceptable for a probe and the same for every arm.

In [ ]:
def score_swing(model: rfdetr.RFDETR, predictor: RFDETRInference, arm: Arm) -> float:
    """Return the mean standard deviation, across sub-patch shifts, of the score of each confident detection."""
    side = model.model_config.resolution
    patch = model.model_config.patch_size
    shifts = [patch * k // 4 for k in range(4)]
    swings = []
    for image in pool[:N_STABILITY_IMAGES]:
        # Clamp: resampling can overshoot [0, 1] by a rounding error, which predict rejects.
        base = resize(to_tensor(image), [side, side], antialias=True).clamp(0.0, 1.0)
        per_shift = []
        for shift in shifts:
            moved = torch.nn.functional.pad(base, (shift, 0, shift, 0))[:, :side, :side]
            per_shift.append(
                (shift, predictor.predict(moved, threshold=0.05, include_source_image=False, tta=arm.config))
            )
        reference = per_shift[0][1]
        for row in np.flatnonzero(reference.confidence >= 0.3):
            scores = [float(reference.confidence[row])]
            for shift, detections in per_shift[1:]:
                same_class = detections.class_id == reference.class_id[row]
                iou = sv.box_iou_batch(reference.xyxy[row : row + 1], detections.xyxy - shift)[0]
                match = same_class & (iou > 0.6)
                scores.append(float(detections.confidence[match].max()) if match.any() else 0.0)
            swings.append(float(np.std(scores)))
    return float(np.mean(swings)) if swings else float("nan")

## Run the sweep

One row per measurement, each with its settings and the run's provenance. A row is identified by `(kind, model, recipe, views, batch_size, variant)`, so a rerun skips what it already has. Delete the work directory after changing the code or the package version: a row is identified by its settings, not by the source it ran on.

In [ ]:
KEY = ("kind", "model", "recipe", "views", "batch_size", "variant")
done = set()
if RESULTS_PATH.exists():
    done = {tuple(row[k] for k in KEY) for row in map(json.loads, RESULTS_PATH.read_text().splitlines())}
print(f"{len(done)} rows already recorded.")

Write one row to disk the moment it exists, so a later failure keeps everything before it.

In [ ]:
def have(kind: str, model_name: str, arm: Arm, batch_size: int | None = None, variant: str = "eager") -> bool:
    """Return whether the row is already recorded."""
    return (kind, model_name, arm.recipe, arm.views, batch_size, variant) in done


def record(
    kind: str, model_name: str, arm: Arm, batch_size: int | None = None, variant: str = "eager", **values: object
) -> None:
    """Append one result row, with the run's provenance, to ``results.jsonl``."""
    row = {
        "kind": kind,
        "model": model_name,
        "recipe": arm.recipe,
        "views": arm.views,
        "batch_size": batch_size,
        "variant": variant,
        **values,
        **provenance,
    }
    with RESULTS_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, allow_nan=True) + "\n")
    done.add((kind, model_name, arm.recipe, arm.views, batch_size, variant))

Run the loop. For each model: accuracy and latency of every arm; for the shift arms also the amortized latency at `B=8`, peak memory, the stage breakdown and the score swing; then the plain prediction at `B` = each view count, which is what views are compared with on the rows axis; then the plain prediction of an optimized snapshot, the reference a user could run instead. Warnings stay visible, since a warning about a deprecated path changes how a number should be read.

In [ ]:
for model_name in MODELS:
    model = getattr(rfdetr, model_name)()
    patch = model.model_config.patch_size
    assert max(VIEWS) <= patch, f"{model_name}: shift views are at most the patch size ({patch}), got {max(VIEWS)}."
    predictor = RFDETRInference(model)
    arms = SHIFT_ARMS + (OTHER_ARMS if model_name in RECIPE_MODELS else [])
    for arm in tqdm(arms, desc=model_name):
        if not have("accuracy", model_name, arm):
            swing = score_swing(model, predictor, arm) if arm.recipe == "shift" else None
            record("accuracy", model_name, arm, **score_ap(predictor, arm, subset), swing=swing)
        batch_sizes = (1, 8) if arm.recipe == "shift" else (1,)
        for batch_size in batch_sizes:
            if not have("latency", model_name, arm, batch_size):
                record("latency", model_name, arm, batch_size, **time_call(predictor, arm, batch_size))
            if arm.recipe == "shift" and not have("memory", model_name, arm, batch_size):
                record("memory", model_name, arm, batch_size, peak_mb=peak_memory_mb(predictor, arm, batch_size))
        if arm.recipe == "shift" and not have("stages", model_name, arm, 1):
            record("stages", model_name, arm, 1, **time_stages(model, predictor, arm))
    for batch_size in VIEWS[1:]:
        if not have("latency", model_name, PLAIN, batch_size):
            record("latency", model_name, PLAIN, batch_size, **time_call(predictor, PLAIN, batch_size))
    if not have("latency", model_name, Arm("plain-optimized", 1, None), 1):
        snapshot = model.inference()
        snapshot_images = pool[:1]
        snapshot_arm = Arm("plain-optimized", 1, None)
        timing = measure_latency(
            lambda: snapshot.predict(snapshot_images, threshold=LATENCY_THRESHOLD, include_source_image=False),
            label="plain-optimized",
            device=device_kind,
            warmup=WARMUP,
            runs=RUNS,
        )
        record("latency", model_name, snapshot_arm, 1, call_ms=timing.median_ms, call_p95_ms=timing.p95_ms)
        del snapshot
    del predictor, model
    if device_kind == "cuda":
        torch.cuda.empty_cache()

## Cheap runtime switches

Two switches change speed without touching the model: TF32 matrix multiplies (`torch.set_float32_matmul_precision("high")`) and cuDNN autotuning (`torch.backends.cudnn.benchmark`). They can move scores slightly, so each is checked for AP as well as latency, on the first `N_SWITCH_AP_IMAGES` images, at one and four shift views. The default row repeats the eager measurement on the same images so the comparison is paired. Both settings are restored afterwards.

In [ ]:
SWITCH_ARMS = (PLAIN, Arm("shift", 4, TTAConfig(shift=4)))
SWITCHES = {
    "default": {"matmul": "highest", "cudnn_benchmark": False},
    "tf32": {"matmul": "high", "cudnn_benchmark": False},
    "tf32+cudnn": {"matmul": "high", "cudnn_benchmark": True},
}
switch_subset = dataclasses.replace(subset, image_ids=subset.image_ids[:N_SWITCH_AP_IMAGES])
saved = (torch.get_float32_matmul_precision(), torch.backends.cudnn.benchmark)
for model_name in [m for m in SWITCH_MODELS if m in MODELS]:
    model = getattr(rfdetr, model_name)()
    predictor = RFDETRInference(model)
    for variant, setting in SWITCHES.items():
        torch.set_float32_matmul_precision(setting["matmul"])
        torch.backends.cudnn.benchmark = setting["cudnn_benchmark"]
        for arm in SWITCH_ARMS:
            if not have("accuracy", model_name, arm, None, variant):
                record(
                    "accuracy", model_name, arm, None, variant, **score_ap(predictor, arm, switch_subset), swing=None
                )
            for batch_size in (1, 8):
                if not have("latency", model_name, arm, batch_size, variant):
                    record("latency", model_name, arm, batch_size, variant, **time_call(predictor, arm, batch_size))
    del predictor, model
torch.set_float32_matmul_precision(saved[0])
torch.backends.cudnn.benchmark = saved[1]

## Read the results back

Load the file, not the loop's variables, so the analysis below also runs on a results file from an earlier session. Keep the row kinds apart, since they carry different values.

In [ ]:
rows = pd.DataFrame(map(json.loads, RESULTS_PATH.read_text().splitlines()))
index = ["model", "recipe", "views", "variant"]
eager = rows[rows["variant"] == "eager"]
# Accuracy rows have no batch size, so the column holds NaN until they are filtered out.
accuracy = (
    eager[eager["kind"] == "accuracy"].set_index(index)[["box_ap", "box_ap50", "fold_box_ap", "swing"]].sort_index()
)
latency = (
    eager[eager["kind"] == "latency"]
    .astype({"batch_size": int})
    .set_index(index + ["batch_size"])[["call_ms", "call_p95_ms"]]
    .sort_index()
)
memory = (
    eager[eager["kind"] == "memory"]
    .astype({"batch_size": int})
    .set_index(index + ["batch_size"])[["peak_mb"]]
    .sort_index()
)
stage_columns = ["place views", "forward", "map back", "decode", "fuse", "other", "total"]
stages = eager[eager["kind"] == "stages"].set_index(index)[[c for c in stage_columns if c in eager.columns]]
models_in_order = [m for m in MODELS if m in set(rows["model"])]

Show the accuracy table for the shift recipe. AP is in percent. `gain` is the model's AP at that view count minus its `V=1` AP on all images; `gain_se` is the standard error of the same gain computed fold by fold, so a gain well inside it is not evidence of an improvement. Each fold is a separate set of images, and the gain is paired: the plain prediction is scored on the same images.

In [ ]:
def paired_gain(frame: pd.DataFrame) -> pd.DataFrame:
    """Add the AP gain over the plain prediction of the same model, and its fold-to-fold standard error."""
    out = frame.copy()
    out["box_ap"] *= 100
    base = frame.xs(("shift", 1), level=("recipe", "views"))
    gains, errors = [], []
    for (model_name, _, _, _), row in frame.iterrows():
        plain = base.loc[(model_name, "eager")]
        fold_gain = (np.array(row["fold_box_ap"]) - np.array(plain["fold_box_ap"])) * 100
        gains.append(row["box_ap"] * 100 - plain["box_ap"] * 100)
        errors.append(fold_gain.std(ddof=1) / np.sqrt(len(fold_gain)))
    out["gain"], out["gain_se"] = gains, errors
    return out.drop(columns=["fold_box_ap", "box_ap50"])


gains = paired_gain(accuracy)
display(gains.round(3))

Show the latency table. `call_ms` is the median time of one call. The ratio divides a row by the same model's plain call at one image, so it reads as how many plain calls a call costs: 1.0 is free and `V` is the cost of running the network `V` times in sequence.

In [ ]:
plain_call = latency.xs(("shift", 1, 1), level=("recipe", "views", "batch_size"))["call_ms"]
cost = latency.copy()
cost["ratio_vs_plain"] = [row.call_ms / plain_call.loc[(key[0], key[3])] for key, row in cost.iterrows()]
display(cost.round(2))

## Plot box AP against latency, one curve per model

The first plot is the one to read first. Each curve is one model size; each point on it is the number of views, labelled `V` (1, 2, 4, 6, 8), at one image per call, so the x axis is the latency of one prediction. The leftmost point of a curve is the plain prediction. Moving along a curve costs latency and buys AP. Comparing curves answers whether views of a small model beat a bigger model's plain prediction: a point above and to the left of another is better on both axes. The cross is each model's plain prediction through an optimized snapshot, the reference a user could run without views.

In [ ]:
colors = {name: plt.cm.viridis(i / max(1, len(models_in_order) - 1)) for i, name in enumerate(models_in_order)}
fig, ax = plt.subplots(figsize=(9, 6))
for name in models_in_order:
    points = [
        (
            latency.loc[(name, "shift", v, "eager", 1), "call_ms"],
            accuracy.loc[(name, "shift", v, "eager"), "box_ap"] * 100,
            v,
        )
        for v in VIEWS
    ]
    ax.plot(
        [p[0] for p in points],
        [p[1] for p in points],
        marker="o",
        color=colors[name],
        label=name.removeprefix("RFDETR"),
    )
    for x, y, v in points:
        ax.annotate(str(v), (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8)
    optimized = latency.xs((name, "plain-optimized"), level=("model", "recipe"))["call_ms"]
    if len(optimized):
        ax.scatter([optimized.iloc[0]], [points[0][1]], marker="x", color=colors[name])
ax.set_xscale("log")
ax.set_xlabel("latency of one call, one image (ms, log)")
ax.set_ylabel("box AP, % (COCO val2017)")
ax.legend(title="model", ncol=2)
ax.grid(alpha=0.3)
plt.show()

How to read it: the labels are views per image. A curve that rises steeply and then flattens has hit diminishing returns at the label where it bends; compare that label with the gain and `gain_se` in the table before calling a bend real. If the plain point of a larger model sits above and to the left of a smaller model's last point, the larger model is the better use of the latency. On a device without CUDA the views do not batch, so every curve is stretched to the right.

## Plot the gain and the cost against the number of views

Two aligned panels with one shared x axis, the number of views per image. The left panel is the paired AP gain over the plain prediction with its fold-to-fold standard error, so the spread of the data is on the plot. The right panel is the latency of one call at one image per call. Together they show where an extra view stops paying.

In [ ]:
fig, (ax_gain, ax_latency) = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
shift_gains = gains.xs(("shift", "eager"), level=("recipe", "variant"))
for name in models_in_order:
    group = shift_gains.loc[name]
    ax_gain.errorbar(
        group.index,
        group["gain"],
        yerr=group["gain_se"],
        marker="o",
        color=colors[name],
        label=name.removeprefix("RFDETR"),
        capsize=2,
    )
    calls = latency.xs((name, "shift", "eager", 1), level=("model", "recipe", "variant", "batch_size"))["call_ms"]
    ax_latency.plot(calls.index, calls.values, marker="o", color=colors[name])
ax_gain.axhline(0, color="gray", linewidth=0.8)
ax_gain.set_ylabel("box AP gain over plain, points")
ax_latency.set_ylabel("latency of one call, one image (ms)")
for ax in (ax_gain, ax_latency):
    ax.set_xscale("log", base=2)
    ax.set_xticks(VIEWS, labels=[str(v) for v in VIEWS])
    ax.set_xlabel("views per image")
    ax.grid(alpha=0.3)
ax_gain.legend(ncol=2, fontsize=8)
plt.show()

How to read it: an error bar that covers zero means the gain is not distinguishable from the fold-to-fold spread. On a batching device the right panel stays flat while the batch is launch-bound and then rises once the forward pass fills the device; the view count where it starts to rise is where views stop being free.

## Plot views against batch images on the same rows axis

A call with `B` images and `V` views pushes `B * V` rows through the network. This plot puts both on one axis, the number of rows. The blue line is `B` plain images at one view each, which is ordinary batching. The orange line is one image with `V` views. They start at the same point and carry the same number of forward rows, so the gap between them is what views cost beyond batching: building them, mapping back and fusing, and the slightly larger canvas a shifted view needs. Small multiples keep every model readable.

In [ ]:
columns = 3
fig, axes = plt.subplots(
    -(-len(models_in_order) // columns),
    columns,
    figsize=(4.2 * columns, 3.2 * -(-len(models_in_order) // columns)),
    squeeze=False,
)
for ax, name in zip(axes.flat, models_in_order):
    plain_batch = latency.xs((name, "shift", 1, "eager"), level=("model", "recipe", "views", "variant"))["call_ms"]
    # Only the batch sizes that equal a view count, so both lines share the same rows axis.
    plain_batch = plain_batch[plain_batch.index.isin(VIEWS)]
    views_call = latency.xs((name, "shift", "eager", 1), level=("model", "recipe", "variant", "batch_size"))["call_ms"]
    ax.plot(plain_batch.index, plain_batch.values, marker="o", label="B images, 1 view")
    ax.plot(views_call.index, views_call.values, marker="o", label="1 image, V views")
    ax.set_title(name.removeprefix("RFDETR"))
    ax.set_xscale("log", base=2)
    ax.set_xticks(VIEWS, labels=[str(v) for v in VIEWS])
    ax.grid(alpha=0.3)
for ax in axes.flat[len(models_in_order) :]:
    ax.set_visible(False)
axes[0][0].legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("forward rows per call (B x V)")
for ax in axes[:, 0]:
    ax.set_ylabel("call latency, ms")
plt.tight_layout()
plt.show()

How to read it: lines that overlap mean a view costs what a batch image costs. The orange line above the blue one is the overhead of views plus their larger canvas; the stage plot below splits it. The B=8 rows in the latency table give the amortized per-image cost for a server that batches images and views together.

## Plot where the time of a call goes

The stage breakdown, from synchronized timers, for one image at the largest view count. The bars split a call into the forward pass and the steps around it: placing the views on canvases, mapping boxes back, decoding the top-k detections to the host, and fusing them. Plain (V=1) calls have only the forward pass and the unattributed rest (reading the image, preprocessing, postprocessing), which is also the floor the views start from.

In [ ]:
largest = VIEWS[-1]
at_largest = stages.xs(("shift", largest, "eager"), level=("recipe", "views", "variant")).reindex(models_in_order)
plain_stage = stages.xs(("shift", 1, "eager"), level=("recipe", "views", "variant")).reindex(models_in_order)
fig, ax = plt.subplots(figsize=(8, 0.5 * len(models_in_order) + 2))
left = np.zeros(len(models_in_order))
for column in [c for c in stage_columns if c != "total" and c in at_largest.columns]:
    values = at_largest[column].fillna(0).values
    ax.barh(range(len(models_in_order)), values, left=left, label=column)
    left += values
ax.set_yticks(range(len(models_in_order)), labels=[m.removeprefix("RFDETR") for m in models_in_order])
ax.set_xlabel(f"ms per call, one image, {largest} views (synchronized timers)")
ax.invert_yaxis()
ax.legend(ncol=3, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.15))
plt.show()

How to read it: the "place views", "map back", "decode" and "fuse" segments together are the overhead of views on top of the network. Their share of the bar is what a faster forward pass (a larger model's cost aside) cannot remove, and is what to optimize next. The table below gives the numbers for every view count.

In [ ]:
display(stages.xs(("shift", "eager"), level=("recipe", "variant")).round(2))

## Plot peak memory against the number of views

Peak extra CUDA memory of one call, from the allocator's high-water mark, at one image and at eight images per call. Memory grows with the rows in flight, so the largest batch with the most views is where an out-of-memory error appears first; the pixel budget that splits views into several forwards is meant to cap this and is set from this plot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, batch_size in zip(axes, (1, 8)):
    for name in models_in_order:
        group = memory.xs((name, "shift", "eager", batch_size), level=("model", "recipe", "variant", "batch_size"))[
            "peak_mb"
        ]
        ax.plot(group.index, group.values, marker="o", color=colors[name], label=name.removeprefix("RFDETR"))
    ax.set_title(f"{batch_size} image{'s' if batch_size > 1 else ''} per call")
    ax.set_xscale("log", base=2)
    ax.set_xticks(VIEWS, labels=[str(v) for v in VIEWS])
    ax.set_xlabel("views per image")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("peak extra memory, MiB")
axes[0].legend(ncol=2, fontsize=8)
plt.show()

How to read it: a flat line means the views were split into forwards of a size that fits the budget, so memory stopped growing at the cost of more forwards. A line that keeps climbing is where the budget does not bind yet. Rows read NaN off CUDA.

## Plot AP gain of each view recipe

The same number of views can be shifts, mirrored copies, zoomed copies or a mix. For the models in `RECIPE_MODELS`, each line is one recipe: the paired AP gain over the plain prediction against the number of views, with the fold-to-fold standard error. The shift line is the one from the first plots. It answers whether spending a view on a mirror or a zoom buys more than spending it on another shift.

In [ ]:
recipe_gains = gains.xs("eager", level="variant")
recipe_models = [m for m in models_in_order if m in RECIPE_MODELS]
fig, axes = plt.subplots(1, len(recipe_models), figsize=(4.6 * len(recipe_models), 4), sharey=True, squeeze=False)
for ax, name in zip(axes[0], recipe_models):
    for recipe, group in recipe_gains.loc[name].groupby("recipe"):
        series = group.droplevel("recipe")
        ax.errorbar(series.index, series["gain"], yerr=series["gain_se"], marker="o", capsize=2, label=recipe)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_title(name.removeprefix("RFDETR"))
    ax.set_xscale("log", base=2)
    ax.set_xticks(VIEWS, labels=[str(v) for v in VIEWS])
    ax.set_xlabel("views per image")
    ax.grid(alpha=0.3)
axes[0][0].set_ylabel("box AP gain over plain, points")
axes[0][-1].legend(fontsize=8)
plt.show()

How to read it: lines that overlap within their error bars are the same recipe for AP; then the cheaper one wins, and zoom views cost more than shift views because their canvas is larger. Mirrored views assume a detection keeps its class when the image is flipped, so a gain from `flip` on COCO does not carry over to a dataset of arrows or turn signs.

## Plot score steadiness against the number of views

The y axis is the mean standard deviation of a confident detection's score across four sub-patch shifts of the same image. It should fall as views are added, because averaging views at several grid phases is what removes the dependence on the grid.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name in models_in_order:
    group = accuracy.xs((name, "shift", "eager"), level=("model", "recipe", "variant"))["swing"]
    ax.plot(group.index, group.values, marker="o", color=colors[name], label=name.removeprefix("RFDETR"))
ax.set_xlabel("views per image")
ax.set_ylabel("mean score std across shifts")
ax.set_xscale("log", base=2)
ax.set_xticks(VIEWS, labels=[str(v) for v in VIEWS])
ax.legend(ncol=2, fontsize=8)
ax.grid(alpha=0.3)
plt.show()

How to read it: the probe uses `N_STABILITY_IMAGES` images and the detections scoring at least 0.3 in the unshifted copy, so a model with few such detections gives a noisy line. Raise the image count before comparing two models. A line that does not fall means the instability this probe measures is not what limits the model on these images.

## Show the runtime switches

AP and latency of the plain prediction and of four shift views for the three runtime settings, on the first `N_SWITCH_AP_IMAGES` images. `ap_change` is the setting minus `default`, in AP points; `speedup` is the default call time divided by the setting's, at one and eight images per call.

In [ ]:
switch_rows = rows[rows["model"].isin(SWITCH_MODELS) & rows["variant"].isin(list(SWITCHES))]
switch_ap = switch_rows[switch_rows["kind"] == "accuracy"].set_index(["model", "views", "variant"])["box_ap"] * 100
switch_ms = (
    switch_rows[switch_rows["kind"] == "latency"]
    .astype({"batch_size": int})
    .set_index(["model", "views", "batch_size", "variant"])["call_ms"]
)
default_ap = switch_ap.xs("default", level="variant")
table = switch_ap.to_frame("box_ap")
table["ap_change"] = switch_ap - default_ap.reindex(table.index.droplevel("variant")).to_numpy()
display(table.round(3))
by_variant = switch_ms.unstack("variant")
speedup = by_variant.rdiv(by_variant["default"], axis=0)
display(speedup.round(3))

How to read it: a speedup above 1 with an `ap_change` inside the fold-to-fold spread above is a free win for that model and device; anything else is not worth a default. TF32 changes the precision of float32 matrix multiplies, so confirm on your own data before relying on it.

## Footer: what produced these numbers

Print the provenance the tables and plots depend on. Paste this footer next to any table you copy into the docs.

In [ ]:
print(
    f"Hardware: {provenance['device']} | rfdetr {provenance['rfdetr']} (commit {provenance['commit']}) | "
    f"rfdetr_plus {provenance['rfdetr_plus']} | torch {provenance['torch']} | supervision {provenance['supervision']} | "
    f"faster-coco-eval {provenance['faster_coco_eval']} | Python {provenance['python']} | "
    f"run started {provenance['started_utc']} UTC | AP on {len(subset.image_ids)} COCO val2017 images at threshold "
    f"{AP_THRESHOLD} in {N_FOLDS} folds, latency at threshold {LATENCY_THRESHOLD}, median of {RUNS} timed calls after "
    f"{WARMUP} warm-up calls."
)